# 📓 Notebook 3 — YOLOv8 Object Detection Training & Evaluation
**Project:** Object Detection and Classification in Low-Light and Occluded Conditions  
**Stage:** Training · Validation · Cross-Dataset Evaluation  

This notebook covers:
1. Setup & GPU verification  
2. Load YOLOv8 model  
3. Train on the enhanced clean dataset  
4. Evaluate: mAP, Precision, Recall, F1  
5. Cross-dataset inference on occ25 / occ50 / occ75  
6. Visualise predictions and confusion matrix  
7. Save best model checkpoint  

## 0 · Install & Import Dependencies

In [ ]:
!pip install ultralytics supervision -q


In [ ]:
import os, random, json, shutil, yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image

from ultralytics import YOLO
import torch

print(f"PyTorch   : {torch.__version__}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 1 · Configuration

In [ ]:
# ──────────────────────────────────────────────────────────────
# CONFIGURATION  (keep in sync with Notebook 1)
# ──────────────────────────────────────────────────────────────

DRIVE_ROOT       = "/content/drive/MyDrive/LowLightDetection"
ENHANCED_DATA_DIR = os.path.join(DRIVE_ROOT, "data", "enhanced")
RUNS_DIR         = os.path.join(DRIVE_ROOT, "runs")

# Which variant to train on
TRAIN_VARIANT    = "clean"

# YOLOv8 model size: n / s / m / l / x
YOLO_MODEL_SIZE  = "s"
YOLO_WEIGHTS     = f"yolov8{YOLO_MODEL_SIZE}.pt"   # pre-trained COCO weights

# Training hyperparameters
IMG_SIZE         = 640
BATCH_SIZE       = 16          # reduce to 8 if OOM on your GPU
EPOCHS           = 100
PATIENCE         = 20          # early stopping patience
LEARNING_RATE    = 0.01
WEIGHT_DECAY     = 0.0005
IOU_THRESHOLD    = 0.5         # for mAP / NMS
CONF_THRESHOLD   = 0.25        # detection confidence threshold

# ExDark classes
EXDARK_CLASSES = [
    "Bicycle", "Boat", "Bottle", "Bus", "Car", "Cat",
    "Chair", "Cup", "Dog", "Motorbike", "People", "Table"
]
NUM_CLASSES = len(EXDARK_CLASSES)

DATASET_VARIANTS = ["clean", "occ25", "occ50", "occ75"]
SPLITS           = ["train", "val", "test"]
SEED             = 42

# Derived
TRAIN_YAML  = os.path.join(ENHANCED_DATA_DIR, TRAIN_VARIANT, "dataset.yaml")
MODEL_SAVE  = os.path.join(DRIVE_ROOT, "models", "yolo_best.pt")
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(MODEL_SAVE), exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("Configuration loaded ✓")
print(f"  Train variant : {TRAIN_VARIANT}")
print(f"  YOLO weights  : {YOLO_WEIGHTS}")
print(f"  Epochs        : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  ImgSz: {IMG_SIZE}")
print(f"  Training yaml : {TRAIN_YAML}")


## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted ✓")


## 3 · Verify dataset.yaml

In [ ]:
def load_and_verify_yaml(yaml_path: str) -> dict:
    """Load YOLO dataset YAML and print a summary."""
    assert os.path.exists(yaml_path), f"YAML not found: {yaml_path}"
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)

    print("dataset.yaml contents:")
    for k, v in cfg.items():
        print(f"  {k}: {v}")

    # Verify image directories exist
    root = cfg.get("path", "")
    for split_key in ["train", "val", "test"]:
        split_path = os.path.join(root, cfg.get(split_key, ""))
        exists = os.path.isdir(split_path)
        print(f"  [{split_key}] {split_path}  →  {'✓' if exists else '✗ MISSING'}")

    return cfg

yaml_cfg = load_and_verify_yaml(TRAIN_YAML)


## 4 · Load YOLOv8 Model

In [ ]:
def load_yolo_model(weights: str = YOLO_WEIGHTS) -> YOLO:
    """Load YOLOv8 model from Ultralytics (downloads if not cached)."""
    model = YOLO(weights)
    print(f"Model loaded: {weights}")
    # Count parameters
    n_params = sum(p.numel() for p in model.model.parameters())
    print(f"Parameters  : {n_params:,}")
    return model

model = load_yolo_model()


## 5 · Training

In [ ]:
def train_yolo(model: YOLO,
               data_yaml: str,
               epochs: int     = EPOCHS,
               img_size: int   = IMG_SIZE,
               batch: int      = BATCH_SIZE,
               patience: int   = PATIENCE,
               lr0: float      = LEARNING_RATE,
               wd: float       = WEIGHT_DECAY,
               project: str    = RUNS_DIR,
               name: str       = "yolov8_exdark",
               seed: int       = SEED) -> object:
    """
    Train YOLOv8 with the given configuration.

    Returns
    -------
    Ultralytics Results object
    """
    results = model.train(
        data          = data_yaml,
        epochs        = epochs,
        imgsz         = img_size,
        batch         = batch,
        patience      = patience,
        lr0           = lr0,
        weight_decay  = wd,
        project       = project,
        name          = name,
        seed          = seed,
        exist_ok      = True,
        # Augmentation flags (built into ultralytics)
        hsv_h         = 0.015,   # hue augment
        hsv_s         = 0.7,     # saturation augment
        hsv_v         = 0.4,     # value augment
        degrees       = 5.0,     # rotation
        translate     = 0.1,
        scale         = 0.5,
        shear         = 2.0,
        flipud        = 0.0,
        fliplr        = 0.5,
        mosaic        = 1.0,
        mixup         = 0.1,
        copy_paste    = 0.1,
        # Logging
        plots         = True,
        save          = True,
        save_period   = 10,      # save checkpoint every N epochs
        verbose       = True,
    )
    return results

print("Starting YOLOv8 training …  (this will take a while on GPU)")
train_results = train_yolo(model, TRAIN_YAML)
print("Training complete ✓")


## 6 · Load Best Checkpoint & Copy to Drive

In [ ]:
def find_best_checkpoint(runs_dir: str, run_name: str = "yolov8_exdark") -> str:
    """Find the best.pt checkpoint from the most recent training run."""
    run_dir  = os.path.join(runs_dir, run_name)
    best_pt  = os.path.join(run_dir,  "weights", "best.pt")
    assert os.path.exists(best_pt), f"Checkpoint not found: {best_pt}"
    return best_pt

best_ckpt = find_best_checkpoint(RUNS_DIR)
print(f"Best checkpoint: {best_ckpt}")

# Copy to permanent Drive location
shutil.copy2(best_ckpt, MODEL_SAVE)
print(f"Saved to Drive : {MODEL_SAVE}")

# Reload for evaluation
best_model = YOLO(MODEL_SAVE)
print("Best model reloaded ✓")


## 7 · Evaluation on Validation Set

In [ ]:
def evaluate_model(model: YOLO,
                   data_yaml: str,
                   img_size: int  = IMG_SIZE,
                   conf: float    = CONF_THRESHOLD,
                   iou: float     = IOU_THRESHOLD,
                   split: str     = "val") -> dict:
    """
    Run YOLOv8 validation and return a metrics dict.

    Returns
    -------
    dict with keys: mAP50, mAP50_95, precision, recall, f1,
                    class_map  (per-class mAP50)
    """
    val_results = model.val(
        data    = data_yaml,
        imgsz   = img_size,
        conf    = conf,
        iou     = iou,
        split   = split,
        verbose = True,
    )

    box = val_results.box
    metrics = {
        "mAP50"    : float(box.map50),
        "mAP50_95" : float(box.map),
        "precision": float(box.mp),
        "recall"   : float(box.mr),
        "f1"       : 2 * float(box.mp) * float(box.mr) /
                     (float(box.mp) + float(box.mr) + 1e-8),
        "class_map": {EXDARK_CLASSES[i]: float(v)
                      for i, v in enumerate(box.maps)
                      if i < len(EXDARK_CLASSES)},
    }
    return metrics

print("Evaluating on validation set …")
val_metrics = evaluate_model(best_model, TRAIN_YAML, split="val")

print("\n── Validation Metrics ──────────────────────")
for k, v in val_metrics.items():
    if k != "class_map":
        print(f"  {k:<15}: {v:.4f}")
print("\n── Per-class mAP50 ─────────────────────────")
for cls, v in val_metrics["class_map"].items():
    print(f"  {cls:<15}: {v:.4f}")


## 8 · Plot Training Curves

In [ ]:
def plot_training_curves(runs_dir: str,
                          run_name: str = "yolov8_exdark",
                          figsize: tuple = (16, 10)) -> None:
    """Read results.csv produced by Ultralytics and plot training curves."""
    import pandas as pd
    csv_path = os.path.join(runs_dir, run_name, "results.csv")
    if not os.path.exists(csv_path):
        print(f"results.csv not found at {csv_path}")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # remove leading spaces

    metrics_pairs = [
        ("train/box_loss", "val/box_loss",        "Box Loss"),
        ("train/cls_loss", "val/cls_loss",        "Class Loss"),
        ("train/dfl_loss", "val/dfl_loss",        "DFL Loss"),
        ("metrics/mAP50(B)", "metrics/mAP50-95(B)", "mAP"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.flatten()

    for ax, (train_col, val_col, title) in zip(axes, metrics_pairs):
        if train_col in df.columns:
            ax.plot(df["epoch"], df[train_col], label="Train", color="royalblue")
        if val_col in df.columns:
            ax.plot(df["epoch"], df[val_col],   label="Val",   color="tomato")
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle("YOLOv8 Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

plot_training_curves(RUNS_DIR)


## 9 · Per-Class mAP Bar Chart

In [ ]:
def plot_per_class_map(metrics: dict, figsize: tuple = (12, 5)) -> None:
    """Bar chart of per-class mAP50."""
    cls_map = metrics["class_map"]
    classes  = list(cls_map.keys())
    values   = list(cls_map.values())

    colors = plt.cm.RdYlGn([v for v in values])
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(classes, values, color=colors, edgecolor="black", linewidth=0.5)

    ax.axhline(metrics["mAP50"], color="navy", linestyle="--",
               linewidth=1.5, label=f"mAP50 = {metrics['mAP50']:.3f}")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("mAP50"); ax.set_xlabel("Class")
    ax.set_title("Per-Class mAP50 — Validation Set", fontsize=13, fontweight="bold")
    plt.xticks(rotation=35, ha="right"); ax.legend(); ax.grid(axis="y", alpha=0.3)

    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.show()

plot_per_class_map(val_metrics)


## 10 · Cross-Dataset Evaluation (occ25 / occ50 / occ75)

In [ ]:
def cross_dataset_eval(model: YOLO,
                        enhanced_root: str,
                        variants: list,
                        split: str = "test") -> dict:
    """
    Evaluate the trained model on every dataset variant.
    Returns a dict mapping variant → metrics dict.
    """
    all_metrics = {}
    for variant in variants:
        yaml_path = os.path.join(enhanced_root, variant, "dataset.yaml")
        if not os.path.exists(yaml_path):
            print(f"[SKIP] YAML not found: {yaml_path}")
            continue
        print(f"\nEvaluating on: {variant} / {split}")
        m = evaluate_model(model, yaml_path, split=split)
        all_metrics[variant] = m
        print(f"  mAP50={m['mAP50']:.4f}  P={m['precision']:.4f}  "
              f"R={m['recall']:.4f}  F1={m['f1']:.4f}")
    return all_metrics

cross_metrics = cross_dataset_eval(best_model, ENHANCED_DATA_DIR,
                                   DATASET_VARIANTS, split="test")


## 11 · Cross-Dataset Comparison Plot

In [ ]:
def plot_cross_dataset(cross_metrics: dict, figsize: tuple = (14, 6)) -> None:
    """Group bar chart comparing metrics across dataset variants."""
    metric_keys = ["mAP50", "mAP50_95", "precision", "recall", "f1"]
    variants    = list(cross_metrics.keys())
    x           = np.arange(len(metric_keys))
    width       = 0.8 / len(variants)
    colors      = plt.cm.tab10(np.linspace(0, 0.5, len(variants)))

    fig, ax = plt.subplots(figsize=figsize)
    for i, (variant, color) in enumerate(zip(variants, colors)):
        m      = cross_metrics[variant]
        values = [m.get(k, 0) for k in metric_keys]
        offset = (i - len(variants) / 2 + 0.5) * width
        bars   = ax.bar(x + offset, values, width,
                        label=variant, color=color, edgecolor="black",
                        linewidth=0.5, alpha=0.85)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f"{v:.2f}", ha="center", va="bottom",
                    fontsize=6.5, rotation=45)

    ax.set_xticks(x); ax.set_xticklabels(metric_keys, fontsize=11)
    ax.set_ylim(0, 1.12); ax.set_ylabel("Score")
    ax.set_title("Detection Metrics Across Dataset Variants",
                 fontsize=13, fontweight="bold")
    ax.legend(title="Variant"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

plot_cross_dataset(cross_metrics)


## 12 · Qualitative Predictions — Draw Bounding Boxes

In [ ]:
def draw_yolo_predictions(model: YOLO,
                            image_dir: str,
                            n_images: int    = 8,
                            conf: float      = CONF_THRESHOLD,
                            figsize: tuple   = (20, 10)) -> None:
    """Run inference and draw predicted bounding boxes on sample images."""
    CLASS_COLORS = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))

    imgs = (list(Path(image_dir).glob("*.jpg")) +
            list(Path(image_dir).glob("*.png")))
    imgs = random.sample(imgs, min(n_images, len(imgs)))

    cols = 4; rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols,
                             figsize=(figsize[0], figsize[1] * rows // 2))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, imgs):
        result   = model.predict(str(img_path), conf=conf,
                                 verbose=False)[0]
        rgb      = cv2.cvtColor(cv2.imread(str(img_path)),
                                cv2.COLOR_BGR2RGB)
        h, w     = rgb.shape[:2]

        ax.imshow(rgb)
        for box in result.boxes:
            xyxy  = box.xyxy[0].cpu().numpy()
            cls   = int(box.cls.item())
            score = float(box.conf.item())
            x1, y1, x2, y2 = xyxy
            color = CLASS_COLORS[cls % NUM_CLASSES]
            rect  = mpatches.Rectangle(
                        (x1, y1), x2 - x1, y2 - y1,
                        linewidth=2, edgecolor=color,
                        facecolor="none")
            ax.add_patch(rect)
            label = f"{EXDARK_CLASSES[cls] if cls < NUM_CLASSES else cls} {score:.2f}"
            ax.text(x1, y1 - 4, label, color="white",
                    fontsize=7, fontweight="bold",
                    bbox=dict(facecolor=color, alpha=0.6, pad=1))

        ax.set_title(Path(img_path).name[:25], fontsize=8)
        ax.axis("off")

    for ax in axes[len(imgs):]:
        ax.axis("off")

    plt.suptitle(f"YOLOv8 Predictions (conf ≥ {conf})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

# Show predictions on test set (clean variant)
test_img_dir = os.path.join(ENHANCED_DATA_DIR, "clean", "images", "test")
if os.path.isdir(test_img_dir):
    draw_yolo_predictions(best_model, test_img_dir)
else:
    print("Test image directory not found.")


## 13 · Save Summary Metrics to JSON

In [ ]:
def save_metrics(metrics: dict, path: str) -> None:
    """Serialise metrics dict to JSON for downstream notebooks."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2, default=str)
    print(f"Metrics saved: {path}")

metrics_out = os.path.join(DRIVE_ROOT, "results", "detection_metrics.json")
save_metrics(
    {"val": val_metrics, "cross_dataset": cross_metrics},
    metrics_out
)
print("All detection metrics persisted ✓")


## ✅ Notebook 3 Complete

**What was done:**
- Verified GPU availability and configured all hyperparameters  
- Loaded YOLOv8-S pre-trained on COCO  
- Trained on the CLAHE-enhanced ExDark (clean) dataset  
- Evaluated on val set: mAP50, mAP50-95, Precision, Recall, F1  
- Cross-dataset evaluation on occ25 / occ50 / occ75 test sets  
- Plotted training curves, per-class mAP, and cross-dataset comparison  
- Visualised bounding-box predictions on test images  
- Saved best weights and metrics JSON to Drive  

**Next step →** `4_crop_generation.ipynb` — use YOLO predictions to build the classification dataset.